# UKBB-trained models applied to KPRB

Transport the frozen UK Biobank models onto Kaiser Permanente incident follow-up.

UK Biobank models were trained mid-2023 (shortly after Phase 2 data release) and use clinical and NMR metabolomics data profiled using plasma samples obtained upon participant enrollment. This notebook loads these pre-trained models, and loads KPRB `meta_clin.csv` and `assay.csv`, harmonized KPRB variables to be compatible with UKBB-trained model predictors. ICD10 codes used to train UK Biobank models are provided in Supplementary Table 14 of the associated manuscript. 

Scaling is `self_nmr`: uses training means and standard deviations for clinical predictors and applies them to the KPRB, while self-scaling (using KPRB mean and SD), for NMR predictors. All six specifications for an endpoint are scored on the same complete cases.

COPD is a supplemental absolute C-index panel. It is not in the nine-outcome figure or the delta C-index figure.

In [ ]:
suppressPackageStartupMessages({
  library(dplyr)
  library(ggplot2)
  library(patchwork)
  library(ggh4x)
  library(here)
})

source(here("helpers", "cv_glmnet_oof.r"))
source(here("helpers", "fit_full_cox.r"))
source(here("helpers", "transport_lp.r"))
source(here("helpers", "prep_incident_time_to_event_df.r"))
source(here("helpers", "plot_oof_cindex_primary.r"))
source(here("helpers", "delta_cindex_oof.r"))

SPEC_ORDER <- c("base", "clin", "marker", "base_nmr", "clin_nmr", "marker_nmr")
FIG_DIR <- here("outputs", "figures")
dir.create(FIG_DIR, recursive = TRUE, showWarnings = FALSE)


# Load models

`ukbb_transport_models` is a named list of endpoints, each a named list of the six frozen fits (`base`, `clin`, `marker`, and the three NMR specifications).


In [ ]:
load(here("models", "ukbb_transport_models_2023.RData"))

stopifnot(is.list(ukbb_transport_models))
stopifnot(all(vapply(
  ukbb_transport_models,
  function(fits) identical(names(fits), SPEC_ORDER),
  logical(1)
)))

cat(
  "Loaded", length(ukbb_transport_models), "endpoints x",
  length(SPEC_ORDER), "specifications\n"
)


# Load KPRB data

Clinical predictors and incident outcomes come from `meta_clin.csv`. NMR predictors come from `assay.csv`. The UKBB fits name alanine `Alanine`; the assay file names it `Ala`.


In [ ]:
meta <- read.csv(here("data", "meta_clin.csv"), row.names = 1) %>%
  dplyr::mutate(StudyID = as.character(StudyID))

assay <- read.csv(here("data", "assay.csv"), row.names = 1) %>%
  dplyr::mutate(StudyID = as.character(StudyID)) %>%
  dplyr::rename(Alanine = Ala)

nmr_names <- setdiff(names(assay), "StudyID")

cat(
  "KPRB:", nrow(meta), "participants |",
  length(nmr_names), "NMR predictors\n"
)


# Harmonize predictor names

UKBB fits use the observed lab, without a KPRB missingness indicator or a separate observed-value name. Examples: `hba1c` is `A1C_obs`, `egfr` is `EGFR_obs`, and `fib4_model` is `log2(FIB4_obs)`.

`meta_clin.csv` stores `0` in a lab column when that lab was never measured (`M_* == 0`). Those zeros are set to `NA` before renaming, so the shared complete-case filter drops untested participants instead of scoring them as a lab value of zero.

Smoking is the exception. COPD and the PREVENT outcomes use `M_smoke`, `smk_current`, and `smk_former`. Atrial fibrillation uses the CHARGE-AF smoking columns `M_smoke_af` and `smoking_current_af`, renamed to the fit names `M_smoke` and `smk_current`.

CHD follow-up is `chd_minerva_status` / `chd_minerva_time`.


In [ ]:
as_int <- function(x) as.integer(as.character(x))

# meta_clin stores 0 for an unmeasured lab. UKBB fits expect NA, not a value of 0.
lab_or_na <- function(value, measured) {
  dplyr::if_else(as_int(measured) == 1L, as.numeric(value), NA_real_)
}

endpoint_config <- list(
  t2d = list(
    status = "t2d_status", time = "t2d_time",
    family = "t2d", label = "T2D"
  ),
  ckd = list(
    status = "ckd_status", time = "ckd_time",
    family = "ckd", label = "CKD"
  ),
  masld = list(
    status = "nafld_status", time = "nafld_time",
    family = "masld", label = "MASLD"
  ),
  copd = list(
    status = "copd_status", time = "copd_time",
    family = "copd", label = "COPD"
  ),
  chd = list(
    status = "chd_minerva_status", time = "chd_minerva_time",
    family = "prevent", label = "CHD"
  ),
  hf_cm = list(
    status = "hf_cm_status", time = "hf_cm_time",
    family = "prevent", label = "Heart Failure"
  ),
  acute_mi = list(
    status = "acute_mi_status", time = "acute_mi_time",
    family = "prevent", label = "Acute MI"
  ),
  tia_stroke = list(
    status = "tia_stroke_status", time = "tia_stroke_time",
    family = "prevent", label = "TIA/Stroke"
  ),
  total_cvd = list(
    status = "total_cvd_status", time = "total_cvd_time",
    family = "prevent", label = "Total CVD"
  ),
  afib = list(
    status = "afib_status", time = "afib_time",
    family = "afib", label = "AFib"
  )
)

harmonize_markers <- function(df, family) {
  if (identical(family, "t2d")) {
    return(dplyr::transmute(df, StudyID, hba1c = lab_or_na(A1C_obs, M_a1c)))
  }
  if (identical(family, "ckd")) {
    return(dplyr::transmute(df, StudyID, egfr = lab_or_na(EGFR_obs, M_egfr)))
  }
  if (identical(family, "masld")) {
    return(dplyr::transmute(
      df,
      StudyID,
      fib4_model = log2(lab_or_na(FIB4_obs, M_fib4))
    ))
  }
  if (identical(family, "copd")) {
    return(dplyr::transmute(
      df,
      StudyID,
      M_smoke = as_int(M_smoke),
      smk_current = as_int(smk_current),
      smk_former = as_int(smk_former),
      ldl_direct = lab_or_na(LDL_obs, M_ldl)
    ))
  }
  if (identical(family, "prevent")) {
    return(dplyr::transmute(
      df,
      StudyID,
      total_cholesterol = lab_or_na(TOT_CHOLES_obs, M_tot_choles),
      hdl_cholesterol = lab_or_na(HDL_CHOLES_obs, M_hdl_choles),
      egfr = lab_or_na(EGFR_obs, M_egfr),
      M_smoke = as_int(M_smoke),
      smk_current = as_int(smk_current),
      smk_former = as_int(smk_former),
      ukb_kprb_bp_treatment = as_int(bp_treatment),
      ukb_kprb_statin_treatment = as_int(statin_treatment),
      prev_diab = as_int(diabetes_at_baseline)
    ))
  }
  if (identical(family, "afib")) {
    return(dplyr::transmute(
      df,
      StudyID,
      M_smoke = as_int(M_smoke_af),
      smk_current = as_int(smoking_current_af),
      dbp = DBP_interp,
      ukb_kprb_bp_treatment = as_int(bp_treatment),
      prev_hf = as_int(hf_at_baseline),
      prev_mi = as_int(mi_at_baseline),
      prev_diab = as_int(diabetes_at_baseline),
      race = as_int(race)
    ))
  }
  stop("Unknown predictor family: ", family)
}

build_scoring_frame <- function(endpoint) {
  cfg <- endpoint_config[[endpoint]]
  if (is.null(cfg)) stop("Unknown endpoint: ", endpoint)

  outcomes <- prep_incident_time_to_event_df(
    meta,
    status_col = cfg$status,
    time_col = cfg$time,
    time_filter = paste(cfg$time, "> 0")
  )

  scored <- outcomes %>%
    dplyr::transmute(
      StudyID,
      time = as.numeric(time),
      status = as.integer(status),
      age_at_sample,
      sex = as_int(sex),
      bmi = BMI_interp,
      sbp = SBP_interp
    ) %>%
    dplyr::inner_join(harmonize_markers(outcomes, cfg$family), by = "StudyID") %>%
    dplyr::inner_join(assay, by = "StudyID")

  needed <- unique(unlist(
    lapply(ukbb_transport_models[[endpoint]], function(fit) fit$predictors),
    use.names = FALSE
  ))
  missing <- setdiff(needed, names(scored))
  if (length(missing) > 0L) {
    stop(
      endpoint, " scoring frame is missing predictor(s): ",
      paste(missing, collapse = ", ")
    )
  }

  scored
}


# Transport

Score each frozen fit on KPRB. `shared_complete_cases = TRUE` keeps one cohort per endpoint: a participant must be non-missing for every predictor used by any of the six specifications.


In [ ]:
run_endpoint_transport <- function(endpoint, specs = SPEC_ORDER,
                                     scaling = "self_nmr") {
  cfg <- endpoint_config[[endpoint]]
  cat("\n##########", cfg$label, "##########\n")
  scoring <- build_scoring_frame(endpoint)
  transport_all_specs(
    models = ukbb_transport_models[[endpoint]],
    newdata = scoring,
    specs = specs,
    rename_map = NULL,
    shared_complete_cases = TRUE,
    scaling = scaling,
    nmr_vars = nmr_names,
    verbose = TRUE
  )
}

transport_results <- setNames(
  vector("list", length(endpoint_config)),
  names(endpoint_config)
)

for (endpoint in names(endpoint_config)) {
  transport_results[[endpoint]] <- run_endpoint_transport(endpoint)
}


# Absolute C-index

Nine cardiometabolic outcomes, then a separate COPD panel.


In [ ]:
ckm_labels <- c(
  t2d = "T2D",
  ckd = "CKD",
  masld = "MASLD",
  hf_cm = "Heart Failure",
  acute_mi = "Acute MI",
  chd = "CHD",
  tia_stroke = "TIA/Stroke",
  total_cvd = "Total CVD",
  afib = "AFib"
)

ckm_dumbbells <- prep_transport_results_list(
  transport_results[names(ckm_labels)],
  endpoint_labels = ckm_labels
)
ckm_dumbbells <- ckm_dumbbells[unname(ckm_labels)]

plot_transport_panel <- function(df, title, show_legend = FALSE) {
  p <- plot_oof_cindex(
    df,
    title = title,
    xlim = c(0.5, 0.85),
    marker_label = "SOC",
    type_labels = c("SOC" = "No NMR", "NMR" = "NMR")
  )
  p + ggplot2::theme(legend.position = if (show_legend) "bottom" else "none")
}

title_prefix <- expression(plain(UKBB) %->% plain(KPRB)~":")

panel_title <- function(prefix, ep) {
  prefix_lang <- if (is.expression(prefix)) prefix[[1L]] else prefix
  bquote(.(prefix_lang) ~ .(ep))
}

ckm_plots <- lapply(names(ckm_dumbbells), function(label) {
  plot_transport_panel(
    ckm_dumbbells[[label]],
    title = panel_title(title_prefix, label),
    show_legend = identical(label, "Total CVD")
  )
})

p_transport <- patchwork::wrap_plots(ckm_plots, ncol = 3) +
  patchwork::plot_annotation(
    title = "UKBB-trained Model Generalization to KPRB",
    theme = ggplot2::theme(plot.title = ggplot2::element_text(hjust = 0.5))
  )

options(repr.plot.width = 8, repr.plot.height = 4.25)
p_transport

ggsave(
  file.path(FIG_DIR, "UKBB_TRANSPORT.png"),
  plot = p_transport,
  width = 8,
  height = 4.25,
  dpi = 600
)




In [ ]:
p_copd <- plot_transport_panel(
  prep_cindex_dumbbell_from_specs(transport_results$copd$summary),
  # title = "UKBB -> KPRB: COPD",
  title = expression(plain(UKBB) %->% plain(KPRB)~": COPD"),
  show_legend = TRUE
)

options(repr.plot.width = 3.5, repr.plot.height = 1.75)
p_copd

ggsave(
  file.path(FIG_DIR, "UKBB_TRANSPORT_COPD.png"),
  plot = p_copd,
  width = 3.5,
  height = 1.75,
  dpi = 600
)


# Delta C-index

Paired NMR increments for BASE, CLIN, and SOC, plus CLIN+NMR versus SOC. Benjamini-Hochberg adjustment is within each comparison. COPD is excluded.


In [ ]:
delta_endpoints <- setdiff(names(endpoint_config), "copd")

delta_spec_map <- c(
  base_SOC   = "base",
  base_NMR   = "base_nmr",
  clin_SOC   = "clin",
  clin_NMR   = "clin_nmr",
  marker_SOC = "marker",
  marker_NMR = "marker_nmr"
)

comparison_template <- data.frame(
  model_1 = c("base_SOC", "clin_SOC", "marker_SOC", "marker_SOC"),
  model_2 = c("base_NMR", "clin_NMR", "marker_NMR", "clin_NMR"),
  stringsAsFactors = FALSE
)

prepare_lp <- function(lp_df) {
  x <- as.data.frame(
    lp_df[, c("StudyID", "lp", "time", "status")],
    stringsAsFactors = FALSE
  )
  x$StudyID <- as.character(x$StudyID)
  x$lp <- as.numeric(x$lp)
  x$time <- as.numeric(x$time)
  x$status <- as.integer(x$status)
  if (anyDuplicated(x$StudyID)) {
    x <- x[!duplicated(x$StudyID), , drop = FALSE]
  }
  list(oof_predictions = x)
}

delta_model_results <- list()
delta_comparisons <- list()

for (nm in delta_endpoints) {
  lp_tables <- transport_results[[nm]]$lp_tables[unname(delta_spec_map)]
  n_by_spec <- vapply(lp_tables, nrow, integer(1))
  events_by_spec <- vapply(
    lp_tables,
    function(x) sum(as.integer(x$status) == 1L, na.rm = TRUE),
    integer(1)
  )
  if (length(unique(n_by_spec)) != 1L || length(unique(events_by_spec)) != 1L) {
    stop(
      "Shared complete-case check failed for ", nm,
      ": n = ", paste(n_by_spec, collapse = "/"),
      "; n_events = ", paste(events_by_spec, collapse = "/")
    )
  }

  for (model_label in names(delta_spec_map)) {
    spec <- unname(delta_spec_map[[model_label]])
    delta_model_results[[paste(nm, model_label, sep = "__")]] <- prepare_lp(
      transport_results[[nm]]$lp_tables[[spec]]
    )
  }

  cmp <- comparison_template
  cmp$endpoint <- endpoint_config[[nm]]$label
  cmp$model_1_key <- paste(nm, cmp$model_1, sep = "__")
  cmp$model_2_key <- paste(nm, cmp$model_2, sep = "__")
  delta_comparisons[[nm]] <- cmp
}

delta_comparisons <- dplyr::bind_rows(delta_comparisons)

delta_all_boosts <- run_delta_cindex_batch(
  results = delta_model_results,
  comparisons = data.frame(
    model_1 = delta_comparisons$model_1_key,
    model_2 = delta_comparisons$model_2_key,
    label_model1 = delta_comparisons$model_1,
    label_model2 = delta_comparisons$model_2,
    stringsAsFactors = FALSE
  ),
  strip_first = FALSE,
  complete_case = FALSE,
  cache_dir = NULL,
  parallel = FALSE,
  lp_higher_risk = TRUE
)

delta_all_boosts <- dplyr::bind_cols(
  endpoint = delta_comparisons$endpoint,
  delta_all_boosts
)

delta_all_boosts <- delta_all_boosts %>%
  dplyr::mutate(
    boost = dplyr::case_when(
      model_1 == "base_SOC"   & model_2 == "base_NMR"   ~ "BASE",
      model_1 == "clin_SOC"   & model_2 == "clin_NMR"   ~ "CLIN",
      model_1 == "marker_SOC" & model_2 == "marker_NMR" ~ "SOC",
      model_1 == "marker_SOC" & model_2 == "clin_NMR"   ~ "CLIN+NMR vs SOC",
      TRUE ~ NA_character_
    )
  ) %>%
  dplyr::filter(!is.na(boost)) %>%
  dplyr::group_by(boost) %>%
  dplyr::mutate(FDR = p.adjust(p_value, method = "BH")) %>%
  dplyr::ungroup()

rm(delta_model_results)
invisible(gc(verbose = FALSE))

delta_all_boosts %>%
  dplyr::arrange(boost, endpoint) %>%
  dplyr::select(
    endpoint, boost, model_1, model_2, n, n_events,
    c_index_1, c_index_2, delta_c, ci_lower, ci_upper, p_value, FDR
  )


In [ ]:
plot_delta_c_boost <- function(
    df,
    x_max = 0.25,
    x_by = 0.10,
    x_min_comparison = -0.02,
    annotation_pad = 0.025
) {

  endpoint_order <- c(
    "T2D", "CKD", "MASLD",
    "Heart Failure", "Acute MI", "CHD",
    "TIA/Stroke", "Total CVD", "AFib"
  )

  boost_order <- c(
    "BASE",
    "CLIN",
    "SOC",
    "CLIN+NMR vs SOC"
  )

  plot_df <- df %>%
    mutate(

      boost = case_when(
        model_1 == "base_SOC"   & model_2 == "base_NMR"   ~ "BASE",
        model_1 == "clin_SOC"   & model_2 == "clin_NMR"   ~ "CLIN",
        model_1 == "marker_SOC" & model_2 == "marker_NMR" ~ "SOC",
        model_1 == "marker_SOC" & model_2 == "clin_NMR"   ~ "CLIN+NMR vs SOC",
        TRUE ~ NA_character_
      ),

      boost = factor(
        boost,
        levels = boost_order
      ),

      endpoint = recode(
        endpoint,
        "HF/CM" = "Heart Failure",
        "AcuteMI" = "Acute MI",
        "TotalCVD" = "Total CVD",
        "Afib" = "AFib"
      ),

      endpoint = factor(
        endpoint,
        levels = rev(endpoint_order)
      ),

      p_adj_BH = if ("p_adj_BH" %in% names(.)) p_adj_BH else FDR
    ) %>%
    filter(!is.na(boost))

  plot_df <- plot_df %>%
    mutate(

      sig_x = if_else(
        delta_c >= 0,
        ci_upper + 0.004,
        ci_lower - 0.004
      ),

      sig_hjust = if_else(
        delta_c >= 0,
        0,
        1
      ),

      sig_label = case_when(
        is.na(p_adj_BH)     ~ NA_character_,
        p_adj_BH >= 0.05    ~ "n.s.",
        p_adj_BH < 0.001    ~ "***",
        p_adj_BH < 0.01     ~ "**",
        TRUE                ~ "*"
      )
    )

  short_x_labels <- function(x) {

    out <- rep("", length(x))
    valid <- !is.na(x)

    is_zero <- valid & abs(x) < 1e-10
    out[is_zero] <- "0"

    is_positive <- valid & x > 1e-10
    out[is_positive] <- sub(
      "^0",
      "",
      sprintf("%.2f", x[is_positive])
    )

    is_negative <- valid & x < -1e-10
    out[is_negative] <- paste0(
      "\u2212",
      sub(
        "^0",
        "",
        sprintf("%.2f", abs(x[is_negative]))
      )
    )

    out
  }

  main_breaks <- seq(
    0,
    floor(x_max / x_by) * x_by,
    by = x_by
  )

  g <- ggplot(
    plot_df,
    aes(
      x = delta_c,
      y = endpoint
    )
  ) +

    geom_col(
      width = 0.62,
      fill = "steelblue"
    ) +

    geom_errorbarh(
      aes(
        xmin = ci_lower,
        xmax = ci_upper
      ),
      height = 0.18,
      linewidth = 0.45
    ) +

    geom_text(
      data = plot_df %>%
        filter(!is.na(sig_label)),

      aes(
        x = sig_x,
        label = sig_label,
        hjust = sig_hjust
      ),

      size = 2.7,
      vjust = 0.7
    ) +

    geom_vline(
      xintercept = 0,
      linewidth = 0.25,
      colour = "black"
    ) +

    facet_wrap(
      ~ boost,
      nrow = 1,
      scales = "free_x"
    ) +

    ggh4x::facetted_pos_scales(
      x = list(

        boost != "CLIN+NMR vs SOC" ~
          scale_x_continuous(
            limits = c(
              0,
              x_max + annotation_pad
            ),

            breaks = main_breaks,

            labels = short_x_labels,

            expand = expansion(
              mult = c(0, 0)
            )
          ),

        boost == "CLIN+NMR vs SOC" ~
          scale_x_continuous(
            limits = c(
              -0.06,
              # -0.045,
              x_max + annotation_pad
            ),

            breaks = seq(
              0,
              x_max,
              by = x_by
            ),

            labels = short_x_labels,

            expand = expansion(
              mult = c(0, 0)
            )
          )
      )
    ) +

    scale_y_discrete(
      drop = FALSE,
      expand = expansion(
        add = c(0.50, 0.55)
      )
    ) +

    labs(
      x = expression(Delta~"C-index (95% CI)"),
      y = NULL,
      title = expression(
        plain(UKBB) %->% plain(KPRB)~"(Incremental NMR Improvement)"
      )
    ) +

    theme_bw(
      base_size = 11
    ) +

    theme(

      strip.background = element_rect(
        fill = "grey92",
        colour = NA
      ),

      strip.text = element_text(
        face = "plain",
        size = 9.5,
        margin = margin(
          t = 4,
          r = 2,
          b = 4,
          l = 2
        )
      ),

      panel.border = element_blank(),

      panel.grid.major.y = element_blank(),

      panel.grid.major.x = element_line(
        colour = "grey90",
        linewidth = 0.3
      ),

      panel.grid.minor = element_blank(),

      panel.spacing.x = grid::unit(
        0.35,
        "cm"
      ),

      axis.line.x = element_line(
        colour = "black",
        linewidth = 0.4
      ),

      axis.ticks.x = element_line(
        colour = "black",
        linewidth = 0.35
      ),

      axis.ticks.y = element_blank(),

      axis.text.x = element_text(
        size = 8,
        colour = "black",
        margin = margin(t = 2)
      ),

      axis.text.y = element_text(
        size = 8.7,
        colour = "black",
        margin = margin(r = 3)
      ),

      axis.title.x = element_text(
        size = 10.5,
        margin = margin(t = 5)
      ),

      plot.title = element_text(
        hjust = 0.5,
        face = "plain",
        size = 12,
        margin = margin(
          b = 6
        )
      ),

      plot.margin = margin(
        t = 3,
        r = 5,
        b = 2,
        l = 3
      )
    )

  return(g)
}

options(repr.plot.height = 2.5, repr.plot.width = 7)

p_delta <- plot_delta_c_boost(
  df = delta_all_boosts,
  x_max = 0.25,
  x_by = 0.10,
  x_min_comparison = -0.02,
  annotation_pad = 0.025
)

p_delta

ggsave(
  file.path(FIG_DIR, "UKBB_TRANSPORT_DELTA.png"),
  plot = p_delta,
  width = 7,
  height = 2.25,
  dpi = 600
)

In [ ]:
library(ggbreak)

fig2c_endpoint_order <- c(
  "T2D", "CKD", "MASLD",
  "Heart Failure", "Acute MI", "CHD",
  "TIA/Stroke", "Total CVD", "AFib"
)

plot_df_fig2c <- delta_all_boosts %>%
  dplyr::filter(model_1 == "marker_SOC", model_2 == "marker_NMR") %>%
  dplyr::mutate(
    endpoint = factor(endpoint, levels = rev(fig2c_endpoint_order)),
    sig_x = dplyr::if_else(delta_c >= 0, ci_upper + 0.0015, ci_lower - 0.001),
    sig_hjust = dplyr::if_else(delta_c >= 0, 0, 1),
    sig_label = dplyr::case_when(
      is.na(FDR)  ~ NA_character_,
      FDR >= 0.05 ~ "n.s.",
      FDR < 0.001 ~ "***",
      FDR < 0.01  ~ "**",
      TRUE        ~ "*"
    )
  )

options(repr.plot.height = 2.3, repr.plot.width = 3.25)

p_fig2c <- ggplot2::ggplot(plot_df_fig2c, ggplot2::aes(x = delta_c, y = endpoint)) +
  ggplot2::geom_col(width = 0.62, fill = "steelblue") +
  ggplot2::geom_errorbarh(
    ggplot2::aes(xmin = ci_lower, xmax = ci_upper),
    height = 0.18, linewidth = 0.45
  ) +
  ggplot2::geom_text(
    data = dplyr::filter(plot_df_fig2c, !is.na(sig_label)),
    ggplot2::aes(x = sig_x, label = sig_label, hjust = sig_hjust),
    size = 2.7, vjust = 0.7
  ) +
  ggplot2::geom_vline(xintercept = 0, linewidth = 0.25, colour = "black") +
  ggplot2::scale_x_continuous(
    limits = c(-0.006, 0.205),
    breaks = c(0, 0.01, 0.02, 0.03, 0.04, 0.05, 0.20),
    labels = c("0", ".01", ".02", ".03", ".04", ".05", ".20"),
    expand = ggplot2::expansion(mult = c(0, 0))
  ) +
  ggbreak::scale_x_break(c(0.05, 0.20), scales = 0.20, space = 0.35, expand = FALSE) +
  ggplot2::scale_y_discrete(drop = FALSE, expand = ggplot2::expansion(add = c(0.50, 0.55))) +
  ggplot2::labs(
    x = expression(Delta~"C-index (95% CI)"),
    y = NULL,
    title = expression(plain(UKBB) %->% plain(KPRB)~"(SOC+NMR)")
  ) +
  ggplot2::theme_bw(base_size = 11) +
  ggplot2::theme(
    panel.border = ggplot2::element_blank(),
    axis.line = ggplot2::element_line(colour = "black", linewidth = 0.4),
    panel.grid.major.y = ggplot2::element_blank(),
    panel.grid.major.x = ggplot2::element_line(colour = "grey90", linewidth = 0.3),
    panel.grid.minor = ggplot2::element_blank(),
    axis.ticks.x = ggplot2::element_line(colour = "black", linewidth = 0.35),
    axis.ticks.y = ggplot2::element_blank(),
    axis.text.x = ggplot2::element_text(size = 8, colour = "black", margin = ggplot2::margin(t = 2)),
    axis.text.x.top = ggplot2::element_blank(),
    axis.ticks.x.top = ggplot2::element_blank(),
    axis.text.y = ggplot2::element_text(size = 11, colour = "black", margin = ggplot2::margin(r = 3)),
    axis.title.x = ggplot2::element_text(size = 11, margin = ggplot2::margin(t = 5)),
    plot.title = ggplot2::element_text(hjust = 0.9, face = "plain", size = 14, margin = ggplot2::margin(b = -8)),
    plot.margin = ggplot2::margin(t = 1, r = 5, b = 2, l = 3)
  )

p_fig2c

ggsave(
  file.path(FIG_DIR, "FIG_2C.png"),
  plot = p_fig2c,
  width = 3,
  height = 2.3,
  dpi = 600
)
 


# COPD delta C-index

All four NMR comparisons for COPD. Benjamini-Hochberg adjustment is across these four tests.
 

In [ ]:
copd_model_results <- lapply(names(delta_spec_map), function(model_label) {
  spec <- unname(delta_spec_map[[model_label]])
  prepare_lp(transport_results$copd$lp_tables[[spec]])
})
names(copd_model_results) <- paste("copd", names(delta_spec_map), sep = "__")

delta_copd <- run_delta_cindex_batch(
  results = copd_model_results,
  comparisons = data.frame(
    model_1 = paste("copd", comparison_template$model_1, sep = "__"),
    model_2 = paste("copd", comparison_template$model_2, sep = "__"),
    label_model1 = comparison_template$model_1,
    label_model2 = comparison_template$model_2,
    stringsAsFactors = FALSE
  ),
  strip_first = FALSE,
  complete_case = FALSE,
  cache_dir = NULL,
  parallel = FALSE,
  lp_higher_risk = TRUE
) %>%
  dplyr::mutate(p_adj_BH = p.adjust(p_value, method = "BH"))

rm(copd_model_results)

plot_delta_c_single <- function(df, title = NULL, x_max = 0.08, x_by = 0.02,
                                annotation_pad = 0.012) {
  boost_order <- c("BASE", "CLIN", "SOC", "CLIN+NMR vs SOC")
  plot_df <- df %>%
    dplyr::mutate(
      boost = dplyr::case_when(
        model_1 == "base_SOC"   & model_2 == "base_NMR"   ~ "BASE",
        model_1 == "clin_SOC"   & model_2 == "clin_NMR"   ~ "CLIN",
        model_1 == "marker_SOC" & model_2 == "marker_NMR" ~ "SOC",
        model_1 == "marker_SOC" & model_2 == "clin_NMR"   ~ "CLIN+NMR vs SOC",
        TRUE ~ NA_character_
      ),
      boost = factor(boost, levels = rev(boost_order)),
      sig_x = dplyr::if_else(delta_c >= 0, ci_upper + 0.004, ci_lower - 0.004),
      sig_hjust = dplyr::if_else(delta_c >= 0, 0, 1),
      sig_label = dplyr::case_when(
        is.na(p_adj_BH)  ~ NA_character_,
        p_adj_BH >= 0.05 ~ "n.s.",
        p_adj_BH < 0.001 ~ "***",
        p_adj_BH < 0.01  ~ "**",
        TRUE             ~ "*"
      )
    ) %>%
    dplyr::filter(!is.na(boost))

  short_x_labels <- function(x) {
    vapply(x, function(z) {
      if (is.na(z)) return("")
      if (abs(z) < 1e-10) return("0")
      if (z < 0) return(paste0("\u2212", sub("^0", "", sprintf("%.2f", abs(z)))))
      sub("^0", "", sprintf("%.2f", z))
    }, character(1))
  }

  x_min <- min(0, min(plot_df$ci_lower, na.rm = TRUE) - 0.004)

  ggplot2::ggplot(plot_df, ggplot2::aes(x = delta_c, y = boost)) +
    ggplot2::geom_col(width = 0.62, fill = "steelblue") +
    ggplot2::geom_errorbarh(
      ggplot2::aes(xmin = ci_lower, xmax = ci_upper),
      height = 0.18, linewidth = 0.45
    ) +
    ggplot2::geom_text(
      data = dplyr::filter(plot_df, !is.na(sig_label)),
      ggplot2::aes(x = sig_x, label = sig_label, hjust = sig_hjust),
      size = 2.7, vjust = 0.7
    ) +
    ggplot2::geom_vline(xintercept = 0, linewidth = 0.25, colour = "black") +
    ggplot2::scale_x_continuous(
      limits = c(x_min, x_max + annotation_pad),
      breaks = seq(0, floor(x_max / x_by) * x_by, by = x_by),
      labels = short_x_labels,
      expand = ggplot2::expansion(mult = c(0, 0))
    ) +
    ggplot2::scale_y_discrete(drop = FALSE, expand = ggplot2::expansion(add = c(0.50, 0.55))) +
    ggplot2::labs(
      x = expression(Delta~"C-index (95% CI)"),
      y = NULL,
      title = title
    ) +
    ggplot2::theme_bw(base_size = 11) +
    ggplot2::theme(
      panel.border = ggplot2::element_blank(),
      panel.grid.major.y = ggplot2::element_blank(),
      panel.grid.major.x = ggplot2::element_line(colour = "grey90", linewidth = 0.3),
      panel.grid.minor = ggplot2::element_blank(),
      axis.line.x = ggplot2::element_line(colour = "black", linewidth = 0.4),
      axis.ticks.x = ggplot2::element_line(colour = "black", linewidth = 0.35),
      axis.ticks.y = ggplot2::element_blank(),
      axis.text.x = ggplot2::element_text(size = 8, colour = "black", margin = ggplot2::margin(t = 2)),
      axis.text.y = ggplot2::element_text(size = 8.7, colour = "black", margin = ggplot2::margin(r = 3)),
      axis.title.x = ggplot2::element_text(size = 10.5, margin = ggplot2::margin(t = 5)),
      plot.title = ggplot2::element_text(hjust = 0.5, face = "plain", size = 12, margin = ggplot2::margin(b = 6)),
      plot.margin = ggplot2::margin(t = 3, r = 5, b = 2, l = 3)
    )
}

options(repr.plot.height = 1.8, repr.plot.width = 4)
p_copd_delta <- plot_delta_c_single(
  df = delta_copd,
  title = expression(plain(UKBB) %->% plain(KPRB)~": COPD"),
  x_max = 0.08,
  x_by = 0.02,
  annotation_pad = 0.012
)
p_copd_delta

ggsave(
  file.path(FIG_DIR, "UKBB_TRANSPORT_COPD_DELTA.png"),
  plot = p_copd_delta,
  width = 3.5,
  height = 1.25,
  dpi = 600
)
